In [2]:
%pip install -qU langchain-exa exa-py

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 1. EXA 가져오기

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

# print(os.getenv("OPENAI_API_KEY"))
# print(os.getenv("EXA_API_KEY"))

True

# 2. EXA Search Tool 생성

In [4]:
from langchain_core.tools import tool
from exa_py import Exa

exa_client = Exa(api_key=os.getenv("EXA_API_KEY"))

@tool # from langchain_core.tools import tool 에서 가져온 것을 데코레이터로 지정
def exa_web_search(query: str) -> str:
    """
    EXA 웹 검색 도구입니다. 주어진 쿼리를 사용하여 웹에서 정보를 검색합니다.
    
    Args:
        query (str): 검색할 쿼리입니다.
        
    Returns:
        str: 검색 결과를 반환합니다.
    """
    results = exa_client.search_and_contents(
        query=query, # 쿼리를 넣고
        num_results=5, # 몇 개의 결과를 가져올지
        text={"max_characters": 1000}, # 글자수 제한
        highlights={"num_sentences": 3}
    )

    output = []
    for result in results.results:
        output. append(f" ** Title **: {result.title}")
        output. append(f" ** URL **: {result.url}")
        if result.highlights:
            output. append(f" ** Highlights **: {' '.join(result.highlights)}")
            output. append(" --- ")

    return "\n".join(output)

print("EXA Search Tool 생성 완료")

EXA Search Tool 생성 완료


In [5]:
# EXA 검색 테스트
test_results = exa_web_search.invoke({"query": "LangGraph agent framework"})
print("EXA Search Tool 테스트 결과:")
print(test_results[:1000] + "..." if len(test_results) > 1000 else test_results)

EXA Search Tool 테스트 결과:
 ** Title **: LangGraph overview - Docs by LangChain
 ** URL **: https://docs.langchain.com/langgraph
 ** Highlights **: # LangGraph overview
[...]
> Gain control with LangGraph to design agents that reliably handle complex tasks
[...]
Trusted by companies shaping the future of agents-- including Klarna, Uber, J.P. Morgan, and more-- LangGraph is a low-level orchestration framework and runtime for building, managing, and deploying long-running, stateful agents.
[...]
LangGraph is very low-level, and focused entirely on agent orchestration. Before using LangGraph, we recommend you familiarize yourself with some of the components used to build agents, starting with models and tools.
[...]
We will commonly use LangChain components throughout the documentation to integrate models and tools, but you don't need to use LangChain to use LangGraph. If you are just getting started with agents or want a higher-level abstraction, we recommend you use Lang
 --- 
 ** Title **

In [1]:
# %pip install langchain
# %pip install langchain-openai

In [7]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# LLM 설정
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

# 도구 리스트
tools = [exa_web_search]

# 에이전트 생성
langchain_agent = create_agent(llm, tools)

print("Langchain Agent 생성 완료")

Langchain Agent 생성 완료


In [10]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

test_question = "2026년 AI 에이전트 프레임워크 트렌드를 알려줘"

print(f"질문: {test_question}")
print("=" * 50)

for chunk in langchain_agent.stream(
    {"messages": [
        {"role": "system", "content": f"You are a helpful AI assistant with web search capabilities. Today is {today}"},
        {"role": "user", "content": test_question}]},
    stream_mode="updates"):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

질문: 2026년 AI 에이전트 프레임워크 트렌드를 알려줘
step: model
content: [{'type': 'tool_call', 'name': 'exa_web_search', 'args': {'query': '2026 AI 에이전트 프레임워크 트렌드'}, 'id': 'call_MedY6bjPR7CGvWrq8wpaunmk'}, {'type': 'tool_call', 'name': 'exa_web_search', 'args': {'query': 'LangChain 2026 업데이트 에이전트'}, 'id': 'call_meaBrv0r3NzOyPZdmyZ36JyZ'}, {'type': 'tool_call', 'name': 'exa_web_search', 'args': {'query': 'OpenAI agents 2026 framework'}, 'id': 'call_aqwRcwdY6XrhOSfZActt8xFt'}, {'type': 'tool_call', 'name': 'exa_web_search', 'args': {'query': 'autonomous agent frameworks 2026 trends'}, 'id': 'call_g8HIcMhIa08eVXnlbrWCeB8w'}, {'type': 'tool_call', 'name': 'exa_web_search', 'args': {'query': 'retrieval-augmented agents 2026'}, 'id': 'call_z67FG6P879mgSJ2DwpKmveDX'}, {'type': 'tool_call', 'name': 'exa_web_search', 'args': {'query': 'AI agent safety regulation 2026'}, 'id': 'call_9bO9dnXc2DIspDfiHabj4jfp'}]
step: tools
content: [{'type': 'text', 'text': " ** Title **: OpenAI Agents SDK\n ** URL **: https://ope

# 4. LangGraph 상태 정의

LangGraph: 명시적인 상태 관리와 그래프 기반 워크플로우를 제공합니다.

- state: 그래프 전체에서 공유되는 상태
- Node: 상태를 변환하는 함수
- Edge: 노드 간 연결(조건부 가능)
- ToolNode: 도구 실행을 담당하는 prebuilt 노드

In [12]:
from typing import Annotated, Literal
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage

# MessagesState: 메시지 리스트를 자동으로 관리하는 내장 상태
# messages 필드가 자동으로 누적됨

print("MessagesState 구조:")
print(" - messages: Annotated[list, add_messages]")
print(" - 메시지가 자동으로 리스트에 추가됨")

MessagesState 구조:
 - messages: Annotated[list, add_messages]
 - 메시지가 자동으로 리스트에 추가됨


# 5. 그래프 노드 및 엣지 구현

In [27]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

# LLM에 도구 바인딩
llm_with_tools = llm.bind_tools(tools)

# 1. 모델 호출 노드
def call_model(state: MessagesState):
    """LLM을 호출하여 응답 또는 도구 호출 결정"""
    print(" --- CALL MODEL --- ")
    
    # 시스템 프롬프트
    system_message = {
        "role": "system",
        "content": f"""You are a helpful AI assistant with web search capabilities. Today is {today}.
Use exa_web_search for questions requiring current information wiyh citations. Please always respond in Korean."""
    }

    messages = [system_message] + state['messages']
    response = llm_with_tools.invoke(messages)

    return {"messages": [response]}


# 2. 조건부 라우팅 함수
def should_continue(state: MessagesState) -> Literal["tools", "_end_"]:
    last_message = state["messages"][-1]

    # tool_calls가 있으면 도구 실행
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        print(" --- ROUTING TO TOOLS --- ")
        return "tools"

    # 없으면 종료
    print(" --- ROUTING TO END --- ")
    return "__end__"

# 3. ToolNode 생성 (도구 실행 담당)
tool_node = ToolNode(tools)

print("노드 및 라우팅 함수 정의 완료!")

노드 및 라우팅 함수 정의 완료!


In [28]:
# 그래프 조립
workflow = StateGraph(MessagesState)

# 노드 추가
workflow.add_node("call_model", call_model)
workflow.add_node("tools", tool_node)

# 엣지 연결
workflow.add_edge(START, "call_model")

# 조건부 엣지: call_model 후 도구 호출 여부에 따라 분기
workflow.add_conditional_edges(
    "call_model",
    should_continue,
    {
        "tools": "tools",
        "__end__": END
    }
)

# 도구 실행 후 다시 모델 호출
workflow.add_edge("tools", "call_model")

# 그래프 컴파일
langgraph_agent = workflow.compile()

print("LangGraph Tool Call Routing 에이전트 컴파일 완료!")

LangGraph Tool Call Routing 에이전트 컴파일 완료!


# 6. Tool Call Routing 에이전트 실행

In [29]:
test1 = "안녕하세요!"

print(f"질문: {test1}")
print("=" * 50)

result = langgraph_agent.invoke({"messages": [{"role": "user", "content": test1}]})
print(f"답변: {result['messages'][-1].content}")

질문: 안녕하세요!
 --- CALL MODEL --- 
 --- ROUTING TO END --- 
답변: 안녕하세요! 무엇을 도와드릴까요?


In [30]:
test2 = "LangGraph와 CrewAI의 차이점을 알려줘"

print(f"질문: {test2}")
print("=" * 50)

result = langgraph_agent.invoke({"messages": [{"role": "user", "content": test2}]})
print(f"답변: {result['messages'][-1].content}")

질문: LangGraph와 CrewAI의 차이점을 알려줘
 --- CALL MODEL --- 
 --- ROUTING TO TOOLS --- 
 --- CALL MODEL --- 
 --- ROUTING TO END --- 
답변: 요약부터 — 둘 다 LLM/에이전트 중심의 워크플로·오케스트레이션 프레임워크지만 목적과 설계 철학이 다릅니다.

주요 차이점(간단 비교)
- 핵심 목적
  - LangGraph: 복잡한 LLM 기반 워크플로를 "명시적 그래프(노드·엣지·상태)"로 모델링하고 제어하기 위해 설계. 단계별 상태 유지·분기·재시도 등 예측 가능한 흐름 제어에 강함 (LangChain 계열) (참고: LangGraph 설명) (https://huggingface.co/learn/agents-course/ko/unit2/langgraph).
  - CrewAI: 여러 자율 에이전트(“크루”)가 협업해서 문제를 해결하도록 조직하고 운영하는 데 초점. 멀티-에이전트 협업, 도구 연동, 실시간 추적·모니터링·스케일링을 염두에 둔 플랫폼/생태계(참고: CrewAI 소개) (https://docs.crewai.com/ko/introduction, https://crewai.com/).

- 아키텍처·구성 요소
  - LangGraph: 방향성 그래프(DAG/상태 그래프)가 1차 개념 — 각 노드는 LLM 호출·도구·의사결정 등 단계, 엣지는 전환 조건. 상태(state)를 명시적으로 전달·판단 (https://huggingface.co/learn/agents-course/ko/unit2/langgraph).
  - CrewAI: Flows(이벤트 기반 워크플로) + Crews(특정 작업을 수행하는 에이전트 팀). Crews 내부에서 여러 역할의 에이전트가 협업하고 외부 툴(메일, 슬랙 등)과 연결(https://docs.crewai.com/ko/introduction).

- 제어 스타일
  - LangGraph: 결정론적·명시적 제어(어떤 조건에서 어느 노드로 갈지